# Sprint 2 - Advanced Retrieval Lab for HelioDesk

This shared Colab notebook supports the Sprint 2 Campus retrieval lessons. You will start from one HelioDesk support-policy handbook, build a naive RAG index, inspect baseline retrieval, add reranking, combine keyword and semantic retrieval, tune the blend, and use HyDE query rewriting to improve recall.

The business question stays fixed so the retrieval changes are easy to compare:

> A customer's outside reviewer is doing a quarterly controls check and asks the support agent for the workspace records package. The agent wants to paste a download link into the ticket thread so the reviewer can grab the file. What should support verify before anything is shared, and what is the approved delivery path?

The notebook uses the shared `ms-ai-ml-helper-core` package rather than lesson-specific helper code. The visible work is the retrieval evidence: which chunks moved, which policy facts reached context, and what risk remains before this could become a production retriever.


## 1. Install the helper core from GitHub

Run this first in Colab. It uses `%pip` so packages install into the active notebook runtime. The helper core is force-reinstalled from the GitHub `main` branch so a fresh Colab run picks up the current shared code.


In [ ]:
#@title Install helper core from GitHub
%pip install -q --force-reinstall --no-cache-dir "ms-ai-ml-helper-core @ git+https://github.com/richhiey/ai-app-dev_Mod-A.git@main"
%pip install -q "pandas>=2,<3" "pysqlite3-binary>=0.5"

print("Installed helper core from GitHub main.")


## 2. Add your OpenRouter key and imports

Embeddings, reranking, HyDE, and the optional final answer all route through OpenRouter. Store `OPENROUTER_API_KEY` in Colab Secrets when possible; the cell below falls back to a hidden prompt. Never paste an API key into notebook source.


In [ ]:
import os
from getpass import getpass


def load_openrouter_key() -> str:
    key = os.getenv("OPENROUTER_API_KEY")
    if key:
        return key

    try:
        from google.colab import userdata

        key = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        key = None

    if not key:
        key = getpass("OpenRouter API key: ")
    if not key:
        raise RuntimeError("OPENROUTER_API_KEY is required for this notebook's model calls.")

    os.environ["OPENROUTER_API_KEY"] = key
    return key


_ = load_openrouter_key()
print("OpenRouter API key loaded.")


In [ ]:
#@title Hidden setup: imports and notebook helpers { display-mode: "form" }
import json
import re
import textwrap
import uuid
from pathlib import Path
from urllib.request import Request, urlopen

try:
    import pysqlite3
    import sys

    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")
except Exception:
    pass

import pandas as pd
from IPython.display import Markdown, display

from documents import Document, chunk_text
from hybrid import HybridRetriever
from hyde import HyDERewriter
from keyword_search import BM25Retriever
from openrouter import OpenRouterClient, OpenRouterEmbedder
from rerank import OpenRouterReranker
from vector_store import ChromaStore

RAW_POLICY_URL = "https://raw.githubusercontent.com/richhiey/ai-app-dev_Mod-A/main/data/heliodesk-policies.md"
POLICY_TITLE = "HelioDesk support policy handbook"

client = OpenRouterClient(app_title="ai-app-dev-module-a-sprint-2-campus")


def load_policy_text() -> tuple[str, str]:
    candidates = [
        Path("heliodesk-policies.md"),
        Path("data/heliodesk-policies.md"),
        Path("../data/heliodesk-policies.md"),
        Path("assets/heliodesk-policies.md"),
        Path("lessons/ml-app-dev/module-a/campus/sprint-2/assets/heliodesk-policies.md"),
    ]
    if Path("/content").exists():
        candidates.insert(0, Path("/content/heliodesk-policies.md"))

    for candidate in candidates:
        if candidate.exists():
            text = candidate.read_text(encoding="utf-8")
            validate_policy_text(text, str(candidate))
            return text, str(candidate)

    request = Request(RAW_POLICY_URL, headers={"User-Agent": "ms-ai-ml-helper-core-colab"})
    with urlopen(request, timeout=30) as response:
        text = response.read().decode("utf-8")
    validate_policy_text(text, RAW_POLICY_URL)
    if Path("/content").exists():
        Path("/content/heliodesk-policies.md").write_text(text, encoding="utf-8")
    return text, RAW_POLICY_URL


def validate_policy_text(text: str, source: str) -> None:
    if POLICY_TITLE not in text:
        raise RuntimeError(
            f"Loaded policy text from {source}, but it did not contain '{POLICY_TITLE}'."
        )


def short_preview(text: str, limit: int = 260) -> str:
    compact = " ".join(text.split())
    return compact if len(compact) <= limit else compact[: limit - 3] + "..."


def evidence_hits(text: str) -> list[str]:
    lower = text.lower()
    return [label for label, phrase in TARGET_EVIDENCE.items() if phrase.lower() in lower]


def result_rows(results, score_attr: str) -> list[dict[str, object]]:
    rows = []
    for rank, result in enumerate(results, start=1):
        score = getattr(result, score_attr, None)
        rows.append(
            {
                "rank": rank,
                "chunk_id": result.document.id,
                "source": result.document.metadata.get("source", "unknown"),
                "score": None if score is None else round(float(score), 4),
                "evidence_hits": ", ".join(evidence_hits(result.document.text)) or "-",
                "preview": short_preview(result.document.text),
            }
        )
    return rows


def show_results(label: str, results, score_attr: str):
    display(Markdown(f"### {label}"))
    display(pd.DataFrame(result_rows(results, score_attr)))


def top_ids(results) -> list[str]:
    return [result.document.id for result in results]


def coverage(results) -> list[str]:
    found = set()
    for result in results:
        found.update(evidence_hits(result.document.text))
    return sorted(found)


def compare_contexts(named_results: dict[str, list]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "run": name,
                "top_chunk_ids": top_ids(results),
                "evidence_covered": coverage(results),
            }
            for name, results in named_results.items()
        ]
    )


def build_context(results) -> str:
    blocks = []
    for result in results:
        title = result.document.metadata.get("source", "heliodesk-policies.md")
        blocks.append(f"[{result.document.id}] {title}\n{result.document.text}")
    return "\n\n---\n\n".join(blocks)


def generate_grounded_answer(query: str, results) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a HelioDesk support assistant. Answer only from the supplied "
                "policy context. Cite chunk ids in square brackets. If the context is "
                "insufficient, say what is missing."
            ),
        },
        {
            "role": "user",
            "content": f"Question:\n{query}\n\nPolicy context:\n{build_context(results)}",
        },
    ]
    response = client.chat(messages, temperature=0, max_tokens=500)
    return response.content or ""


## 3. Create the HelioDesk retrieval corpus and test queries

The corpus is one policy handbook for a fictional B2B support team. The same reviewer-controls question will be used across the baseline, reranking, hybrid, and HyDE sections so you can compare retrieval changes without changing the business problem.

The parameter values below are intentionally visible. Change one value at a time during the Independent Practice sections, then rerun the dependent cells and inspect which chunks move.


In [ ]:
#@title Parameter panel and corpus setup
FOCUS_QUERY = (
    "A customer's outside reviewer is doing a quarterly controls check and asks the support agent "
    "for the workspace records package. The agent wants to paste a download link into the ticket "
    "thread so the reviewer can grab the file. What should support verify before anything is "
    "shared, and what is the approved delivery path?"
)

CANDIDATE_K = 10
FINAL_CONTEXT_K = 3
DEFAULT_HYBRID_ALPHA = 0.65
TUNING_ALPHAS = [0.35, 0.50, 0.65, 0.80]
HYDE_HYBRID_ALPHA = 0.65
HYDE_CONTEXT_HINT = (
    "HelioDesk is a B2B support-policy assistant. Useful source terms may include "
    "external auditor, written authorization, account owner, public links, ticket comments, "
    "secure customer portal, export files, administrator, and billing manager."
)

TARGET_EVIDENCE = {
    "download_authority": "Only workspace administrators and billing managers can download completed export files",
    "external_auditor_authorization": "support must verify written authorization from the account owner",
    "secure_portal_delivery": "secure customer portal",
    "no_ticket_comment_link": "must not be shared through public links or copied into ticket comments",
}

policy_text, policy_source = load_policy_text()
missing = [label for label, phrase in TARGET_EVIDENCE.items() if phrase.lower() not in policy_text.lower()]
if missing:
    raise RuntimeError(f"Target evidence is missing from the policy source: {missing}")

documents = chunk_text(
    policy_text,
    source_id="HDPOL",
    chunk_size=850,
    overlap=120,
    metadata={"source": "heliodesk-policies.md", "case": "HelioDesk"},
)

print(f"Loaded policy source: {policy_source}")
print(f"Policy characters: {len(policy_text):,}")
print(f"Generated chunks: {len(documents)}")
print(f"CANDIDATE_K={CANDIDATE_K}; FINAL_CONTEXT_K={FINAL_CONTEXT_K}; DEFAULT_HYBRID_ALPHA={DEFAULT_HYBRID_ALPHA}")
print("Focus query:")
print(textwrap.fill(FOCUS_QUERY, width=96))


## 4. Index chunks into ChromaDB

This is the indexing half of naive RAG. Each chunk becomes a stored document with an embedding, which is a numeric representation of text meaning. The cell uses a fresh Chroma directory every run so a restarted Colab runtime does not reuse a half-built SQLite store.


In [ ]:
RUN_ID = uuid.uuid4().hex[:8]
DB_ROOT = Path("/content/module_a_sprint_2_chroma") if Path("/content").exists() else Path(".chroma/module_a_sprint_2")
DB_PATH = DB_ROOT / f"run_{RUN_ID}"
DB_PATH.mkdir(parents=True, exist_ok=False)

embedder = OpenRouterEmbedder(client)
store = ChromaStore(path=DB_PATH, collection_name="heliodesk_support_policies", embedder=embedder)
indexed_count = store.index(documents, batch_size=8)
keyword = BM25Retriever.from_documents(store.all_documents())
hybrid = HybridRetriever(vector_store=store, keyword_retriever=keyword, alpha=0.65)

print(f"Indexed {indexed_count} chunks into ChromaDB at {DB_PATH}.")
print(f"BM25 keyword retriever sees {len(keyword.documents)} chunks from the same corpus.")


## 5. Baseline retrieval

This is the retrieval half of naive RAG. The app embeds the user's question, compares it with stored chunk embeddings, and returns the nearest chunks. Inspect the chunks before trusting any generated answer.


In [ ]:
baseline_candidates = store.semantic_search(FOCUS_QUERY, top_k=CANDIDATE_K)
baseline_context = baseline_candidates[:FINAL_CONTEXT_K]

show_results("Baseline semantic candidates", baseline_candidates, "semantic_score")
display(compare_contexts({"baseline_top_3": baseline_context}))


## 6. Rerank the candidate pool

A reranker receives the original question and the candidate chunks from first-pass retrieval, then scores those candidates again for direct question fit. It does not search the whole handbook, so the useful evidence must already be in the candidate pool.


In [ ]:
reranker = OpenRouterReranker(client)
reranked_context = reranker.rerank(FOCUS_QUERY, baseline_candidates, top_n=FINAL_CONTEXT_K)

show_results("Reranked final context", reranked_context, "rerank_score")
display(
    compare_contexts(
        {
            "semantic_top_3": baseline_context,
            "reranked_top_3": reranked_context,
        }
    )
)


## 7. Add hybrid search

Hybrid retrieval combines two first-pass signals over the same chunks: semantic retrieval through ChromaDB and keyword retrieval through BM25. In the helper core, `alpha` is the semantic weight and `1 - alpha` is the keyword weight.


In [ ]:
hybrid_context = hybrid.search(
    FOCUS_QUERY,
    top_k=FINAL_CONTEXT_K,
    semantic_k=CANDIDATE_K,
    keyword_k=CANDIDATE_K,
    alpha=DEFAULT_HYBRID_ALPHA,
)

show_results(f"Hybrid final context, alpha={DEFAULT_HYBRID_ALPHA}", hybrid_context, "hybrid_score")
display(
    compare_contexts(
        {
            "semantic_top_3": baseline_context,
            "reranked_top_3": reranked_context,
            f"hybrid_alpha_{DEFAULT_HYBRID_ALPHA}": hybrid_context,
        }
    )
)


## 8. Tune the blend

The blend weight is a hypothesis about the query shape. Keep the source document and question fixed, change only `alpha`, and judge the result by retrieved evidence rather than by score values alone.


In [ ]:
blend_rows = []
for alpha in TUNING_ALPHAS:
    tuned = hybrid.search(
        FOCUS_QUERY,
        top_k=FINAL_CONTEXT_K,
        semantic_k=CANDIDATE_K,
        keyword_k=CANDIDATE_K,
        alpha=alpha,
    )
    blend_rows.append(
        {
            "alpha_semantic_weight": alpha,
            "keyword_weight": round(1 - alpha, 2),
            "top_chunk_ids": top_ids(tuned),
            "evidence_covered": coverage(tuned),
        }
    )

display(pd.DataFrame(blend_rows))


## 9. Rewrite the query with HyDE

HyDE means Hypothetical Document Embeddings. A model writes a short hypothetical source-like passage, that passage is embedded, and retrieval searches for real chunks that look similar to it. The hypothetical document is a search key, not evidence for the final answer.


In [ ]:
hyde = HyDERewriter(client).rewrite(
    FOCUS_QUERY,
    context_hint=HYDE_CONTEXT_HINT,
)

print(json.dumps(hyde.model_dump(), indent=2))

hyde_semantic_candidates = store.semantic_search(hyde.hypothetical_document, top_k=CANDIDATE_K)
hyde_hybrid_context = hybrid.search(
    FOCUS_QUERY,
    top_k=FINAL_CONTEXT_K,
    semantic_k=CANDIDATE_K,
    keyword_k=CANDIDATE_K,
    alpha=HYDE_HYBRID_ALPHA,
    semantic_query=hyde.hypothetical_document,
    keyword_query=hyde.rewritten_query,
)

show_results("HyDE semantic candidates", hyde_semantic_candidates, "semantic_score")
show_results(f"HyDE-assisted hybrid final context, alpha={HYDE_HYBRID_ALPHA}", hyde_hybrid_context, "hybrid_score")
display(
    compare_contexts(
        {
            "baseline_top_3": baseline_context,
            f"hybrid_alpha_{DEFAULT_HYBRID_ALPHA}": hybrid_context,
            "hyde_hybrid_top_3": hyde_hybrid_context,
        }
    )
)


## 10. Build the checkpoint defense

A good Sprint 2 retrieval defense names the lever, shows the before-and-after chunks, and explains the remaining risk. Use this final cell to collect the evidence you would cite in a walkthrough, practice note, or checkpoint submission.


In [ ]:
evidence_summary = compare_contexts(
    {
        "naive_semantic": baseline_context,
        "reranked": reranked_context,
        "hybrid_alpha_0.65": hybrid_context,
        "hyde_hybrid": hyde_hybrid_context,
    }
)
display(evidence_summary)

retrieval_defense = [
    {
        "lever": "Naive RAG indexing + retrieval",
        "evidence_question": "Did the semantic top results include the policy facts needed for the answer?",
    },
    {
        "lever": "Reranking",
        "evidence_question": "Did a useful candidate move into the final context after the second judgment step?",
    },
    {
        "lever": "Hybrid retrieval",
        "evidence_question": "Did BM25 plus semantic search cover evidence that either signal missed alone?",
    },
    {
        "lever": "HyDE query rewriting",
        "evidence_question": "Did answer-shaped search text improve recall without becoming unsupported answer evidence?",
    },
]
display(pd.DataFrame(retrieval_defense))

assert indexed_count == len(documents), "The Chroma index count should match the chunk count."
assert baseline_candidates, "Baseline semantic retrieval returned no candidates."
assert reranked_context, "Reranking returned no results."
assert hybrid_context, "Hybrid retrieval returned no results."
assert hyde_hybrid_context, "HyDE-assisted retrieval returned no results."

RUN_FINAL_ANSWER = False
if RUN_FINAL_ANSWER:
    best_context = hyde_hybrid_context
    answer = generate_grounded_answer(FOCUS_QUERY, best_context)
    display(Markdown(answer))
else:
    print("Notebook checks passed.")
    print("Set RUN_FINAL_ANSWER = True in this cell to generate one final grounded answer from the HyDE-assisted context.")


## 11. Assessment workspace: three retrieval tasks

Use this final section after you have run the walkthrough and practice sections above. You will complete exactly three assessment tasks with the same HelioDesk policy corpus, `ChromaStore` index, `BM25Retriever`, `HybridRetriever`, `OpenRouterReranker`, and `HyDERewriter` objects.

Do not edit `heliodesk-policies.md`, generated chunk text, or chunk IDs. Change only the assessment parameters marked in the cells below, then support your decisions with real `HDPOL:chunk-####` evidence.


In [ ]:
ASSESSMENT_TASKS = [
    {
        "task_id": "task_1_reranking_audit",
        "focus": "Run naive semantic retrieval, then rerank a noisy auditor-delivery query.",
        "evidence_to_find": "authorization, no ticket-comment link, secure portal delivery",
    },
    {
        "task_id": "task_2_hybrid_blend",
        "focus": "Tune the hybrid blend for a refund and service-credit query.",
        "evidence_to_find": "14-day renewal review, service-credit exception, billing-operations routing",
    },
    {
        "task_id": "task_3_hyde_recall",
        "focus": "Use HyDE for an Enterprise incident query whose wording differs from the policy.",
        "evidence_to_find": "critical severity, 1 business hour first response, no fix-time promise",
    },
]

display(pd.DataFrame(ASSESSMENT_TASKS))


In [ ]:
ASSESSMENT_QUERIES = {
    "task_1_reranking_audit": (
        "A customer's outside compliance reviewer asks the agent for a workspace records package. "
        "The agent wants to paste a completed export link into the public ticket so the reviewer can download it. "
        "What checks and delivery route should support use?"
    ),
    "task_2_hybrid_blend": (
        "An annual customer renewed, used a renewal-period service credit, and asks another agent "
        "to override the refund rule. What should support say and where should the case go?"
    ),
    "task_3_hyde_recall": (
        "An Enterprise customer says they may have lost production data and need a launch unblocked today. "
        "What priority and response promise should support apply, and what should they avoid promising?"
    ),
}

# Task 1: naive retrieval plus reranking. Change TASK_1_CANDIDATE_K only if your first run misses key evidence.
TASK_1_CANDIDATE_K = 10
TASK_1_FINAL_K = 3

task_1_query = ASSESSMENT_QUERIES["task_1_reranking_audit"]
task_1_candidates = store.semantic_search(task_1_query, top_k=TASK_1_CANDIDATE_K)
task_1_reranked = reranker.rerank(task_1_query, task_1_candidates, top_n=TASK_1_FINAL_K)

show_results("Assessment task 1 - semantic candidates", task_1_candidates, "semantic_score")
show_results("Assessment task 1 - reranked context", task_1_reranked, "rerank_score")
display(compare_contexts({"task_1_semantic_top_3": task_1_candidates[:TASK_1_FINAL_K], "task_1_reranked_top_3": task_1_reranked}))


In [ ]:
# Task 2: hybrid blend tuning. Change TASK_2_ALPHAS if you want to test one extra value.
TASK_2_ALPHAS = [0.35, 0.50, 0.65, 0.80]
TASK_2_FINAL_K = 3

task_2_query = ASSESSMENT_QUERIES["task_2_hybrid_blend"]
task_2_rows = []
task_2_runs = {}
for alpha in TASK_2_ALPHAS:
    results = hybrid.search(
        task_2_query,
        top_k=TASK_2_FINAL_K,
        semantic_k=CANDIDATE_K,
        keyword_k=CANDIDATE_K,
        alpha=alpha,
    )
    task_2_runs[f"alpha_{alpha}"] = results
    task_2_rows.append(
        {
            "alpha": alpha,
            "top_chunk_ids": top_ids(results),
            "evidence_covered": coverage(results),
        }
    )

display(pd.DataFrame(task_2_rows))

TASK_2_CHOSEN_ALPHA = 0.65
show_results(
    f"Assessment task 2 - chosen hybrid context, alpha={TASK_2_CHOSEN_ALPHA}",
    task_2_runs[f"alpha_{TASK_2_CHOSEN_ALPHA}"],
    "hybrid_score",
)


In [ ]:
# Task 3: HyDE recall check. Change TASK_3_HYDE_ALPHA or the hint only after recording the first run.
TASK_3_HYDE_ALPHA = 0.65
TASK_3_FINAL_K = 3
TASK_3_HYDE_HINT = (
    "HelioDesk policy language may include Enterprise, severity-one, critical support tickets, "
    "first response, business hour, data loss, production outage, named escalation contact, "
    "incident commander, and resolution estimate."
)

task_3_query = ASSESSMENT_QUERIES["task_3_hyde_recall"]
task_3_baseline = store.semantic_search(task_3_query, top_k=TASK_3_FINAL_K)
task_3_hyde = HyDERewriter(client).rewrite(task_3_query, context_hint=TASK_3_HYDE_HINT)
print(json.dumps(task_3_hyde.model_dump(), indent=2))

task_3_hyde_context = hybrid.search(
    task_3_query,
    top_k=TASK_3_FINAL_K,
    semantic_k=CANDIDATE_K,
    keyword_k=CANDIDATE_K,
    alpha=TASK_3_HYDE_ALPHA,
    semantic_query=task_3_hyde.hypothetical_document,
    keyword_query=task_3_hyde.rewritten_query,
)

show_results("Assessment task 3 - baseline semantic context", task_3_baseline, "semantic_score")
show_results(f"Assessment task 3 - HyDE-assisted hybrid context, alpha={TASK_3_HYDE_ALPHA}", task_3_hyde_context, "hybrid_score")
display(compare_contexts({"task_3_baseline": task_3_baseline, "task_3_hyde": task_3_hyde_context}))


In [ ]:
ASSESSMENT_SUBMISSION = {
    "task_1_reranking_audit": {
        "selected_context": "reranked",  # choose semantic_top_3 or reranked
        "chunk_ids": [],
        "evidence_note": "",
        "remaining_risk": "",
    },
    "task_2_hybrid_blend": {
        "chosen_alpha": TASK_2_CHOSEN_ALPHA,
        "chunk_ids": [],
        "evidence_note": "",
        "why_not_global_setting": "",
    },
    "task_3_hyde_recall": {
        "hyde_terms_added": [],
        "chunk_ids": [],
        "evidence_note": "",
        "hyde_limitation": "",
    },
}

CHECK_SUBMISSION = False
if CHECK_SUBMISSION:
    for task_id, answer in ASSESSMENT_SUBMISSION.items():
        assert answer.get("chunk_ids"), f"{task_id} needs at least one real HDPOL chunk id."
        assert answer.get("evidence_note", "").strip(), f"{task_id} needs an evidence note."
    print("Assessment submission shape looks complete. Now check the content against the retrieved chunks.")
else:
    display(Markdown("Set `CHECK_SUBMISSION = True` after filling `ASSESSMENT_SUBMISSION`."))
